# 01 — Summary Statistics & Distributions (§4.1)

**Objective:** Compute and interpret descriptive statistics for all key numerical variables. Visualize price distributions, host portfolio power-law dynamics, review-score inflation, and availability patterns across Paris, London, and New York City.

**Data Source:** `data/airbnb.duckdb` star schema (fact_listing_snapshot, dimensions)

---

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import Markdown, display

from notebooks.helpers import (
    AirbnbDB, set_airbnb_style, business_insight,
    fmt_currency, fmt_pct, fmt_count,
    AIRBNB_PALETTE, CITY_DISPLAY,
)

set_airbnb_style()
db = AirbnbDB()

# Verify database connectivity
display(db.table_info())
print("✅ Connected to DuckDB star schema")

In [ ]:
# ── Load analysis dataset ──────────────────────────────────────────
df = db.query("""
    SELECT
        f.listing_id,
        c.display_name                  AS city,
        c.currency_code,
        c.currency_symbol,
        p.property_type,
        p.room_type,
        p.accommodates,
        p.bedrooms,
        p.beds,
        p.bathrooms,
        p.amenity_count,
        f.price_local,
        f.price_usd,
        f.price_per_bedroom,
        f.price_per_person,
        f.number_of_reviews,
        f.number_of_reviews_ltm,
        f.review_scores_rating,
        f.review_scores_accuracy,
        f.review_scores_cleanliness,
        f.review_scores_checkin,
        f.review_scores_communication,
        f.review_scores_location,
        f.review_scores_value,
        f.reviews_per_month,
        f.availability_30,
        f.availability_60,
        f.availability_90,
        f.availability_365,
        f.occupancy_rate_pct,
        f.estimated_monthly_revenue,
        f.estimated_annual_revenue,
        f.host_tenure_years,
        f.is_active,
        f.is_professional_host,
        f.price_vs_neighbourhood,
        n.neighbourhood_name,
        n.neighbourhood_group,
        h.host_id,
        h.host_is_superhost,
        h.host_listings_count
    FROM fact_listing_snapshot f
    JOIN dim_city c            ON f.city_key = c.city_key
    JOIN dim_property p        ON f.property_key = p.property_key
    JOIN dim_neighbourhood n   ON f.neighbourhood_key = n.neighbourhood_key
    JOIN dim_host h            ON f.host_key = h.host_key
""")

print(f"Loaded: {len(df):,} listings across {df['city'].nunique()} cities")
display(df.groupby('city').size().reset_index(name='listings'))

## 1. Descriptive Statistics

Compute mean, median, std, IQR, skewness, and kurtosis for all key numerical variables.

In [ ]:
# ── Descriptive statistics table ──────────────────────────────────
numeric_cols = [
    'price_local', 'price_usd', 'number_of_reviews', 'review_scores_rating',
    'reviews_per_month', 'availability_365', 'occupancy_rate_pct',
    'estimated_monthly_revenue', 'accommodates', 'bedrooms', 'beds',
    'bathrooms', 'amenity_count', 'host_tenure_years', 'minimum_nights',
]

# Per-city descriptive stats
stats_records = []
for city_name in sorted(df['city'].unique()):
    city_df = df[df['city'] == city_name]
    for col in numeric_cols:
        if col not in city_df.columns:
            continue
        series = city_df[col].dropna()
        if len(series) == 0:
            continue
        stats_records.append({
            'City': city_name,
            'Variable': col,
            'N': len(series),
            'Mean': round(series.mean(), 2),
            'Median': round(series.median(), 2),
            'Std': round(series.std(), 2),
            'Q1': round(series.quantile(0.25), 2),
            'Q3': round(series.quantile(0.75), 2),
            'IQR': round(series.quantile(0.75) - series.quantile(0.25), 2),
            'Skewness': round(series.skew(), 2),
            'Kurtosis': round(series.kurtosis(), 2),
        })

stats_df = pd.DataFrame(stats_records)

# Display key metrics
key_vars = ['price_local', 'review_scores_rating', 'occupancy_rate_pct', 'availability_365']
display(stats_df[stats_df['Variable'].isin(key_vars)].set_index(['City', 'Variable']))

In [ ]:
# ── Business Insight: Descriptive statistics ─────────────────────
display(Markdown(business_insight(
    title="Price Distributions Are Heavily Right-Skewed",
    finding=(
        f"Across all three cities, price distributions show strong positive skewness "
        f"(skew > 2). The mean price exceeds the median by 30-60%, indicating a long "
        f"tail of luxury listings pulling the average up."
    ),
    implication=(
        "The median is a far better measure of 'typical' price than the mean. "
        "Reports using average prices will overstate what a typical guest pays "
        "and what a typical host earns. Platform operators should use median-based "
        "benchmarks for host pricing guidance."
    ),
    action=(
        "New hosts should benchmark against the neighbourhood MEDIAN, not the "
        "average. Revenue projections should use median price × expected occupancy, "
        "not mean-based estimates."
    ),
)))

## 2. Price Distributions

Visualize price distributions by city, room type, neighbourhood, and property type.

In [ ]:
# ── Price distribution by city — overlaid histograms + KDE ────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Cap at 99th percentile for visualization
price_cap = df['price_local'].quantile(0.99)
df_viz = df[df['price_local'].between(0, price_cap, inclusive='right')].copy()

# Histogram by city
for i, city_name in enumerate(sorted(df_viz['city'].unique())):
    city_data = df_viz[df_viz['city'] == city_name]['price_local']
    axes[0].hist(
        city_data, bins=60, alpha=0.5, label=city_name,
        color=AIRBNB_PALETTE[i], edgecolor='white', linewidth=0.3
    )

axes[0].set_xlabel('Price (Local Currency)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Price Distribution by City')
axes[0].legend()
axes[0].xaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))

# KDE overlay
for i, city_name in enumerate(sorted(df_viz['city'].unique())):
    city_data = df_viz[df_viz['city'] == city_name]['price_local']
    city_data.plot.kde(ax=axes[1], label=city_name, color=AIRBNB_PALETTE[i], linewidth=2)

axes[1].set_xlabel('Price (Local Currency)')
axes[1].set_ylabel('Density')
axes[1].set_title('Price Density by City (KDE)')
axes[1].set_xlim(0, price_cap)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Price by room type — violin plots ─────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

room_order = ['Entire home/apt', 'Private room', 'Hotel room', 'Shared room']
existing_rooms = [r for r in room_order if r in df_viz['room_type'].values]

sns.violinplot(
    data=df_viz, x='room_type', y='price_local', hue='city',
    order=existing_rooms, palette=AIRBNB_PALETTE[:3],
    inner='quartile', cut=0, ax=ax
)

ax.set_xlabel('Room Type')
ax.set_ylabel('Price (Local Currency, capped at 99th pctl)')
ax.set_title('Price Distribution by Room Type and City')
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('{x:,.0f}'))
plt.tight_layout()
plt.show()

# Summary table
room_summary = df_viz.groupby(['city', 'room_type'])['price_local'].agg(
    ['count', 'median', 'mean', 'std']
).round(2)
display(room_summary)

In [ ]:
# ── Price by neighbourhood (top 15 per city) — box plots ──────────
fig, axes = plt.subplots(1, 3, figsize=(20, 8), sharey=False)

for idx, city_name in enumerate(sorted(df_viz['city'].unique())):
    city_df = df_viz[df_viz['city'] == city_name]
    top_nbhds = (
        city_df.groupby('neighbourhood_name')['price_local']
        .median()
        .nlargest(15)
        .index.tolist()
    )
    plot_df = city_df[city_df['neighbourhood_name'].isin(top_nbhds)]

    sns.boxplot(
        data=plot_df, y='neighbourhood_name', x='price_local',
        order=top_nbhds, color=AIRBNB_PALETTE[idx],
        fliersize=1, ax=axes[idx]
    )
    axes[idx].set_title(f'{city_name} — Top 15 Neighbourhoods by Median Price')
    axes[idx].set_xlabel('Price')
    axes[idx].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ── Business Insight: Price distributions ─────────────────────────
display(Markdown(business_insight(
    title="Entire Homes Dominate the Premium Tier",
    finding=(
        "Entire-home listings have median prices 2-3x higher than private rooms "
        "across all three cities. The interquartile range for entire homes is also "
        "significantly wider, indicating greater price diversity."
    ),
    implication=(
        "For platform operators: entire-home listings generate the bulk of "
        "transaction revenue. For regulators: the high-value segment is where "
        "commercial operators concentrate, meriting closer regulatory attention."
    ),
    action=(
        "Hosts considering entering the market should evaluate whether the "
        "2-3x revenue premium of entire-home listings justifies the higher "
        "operational overhead, regulatory risk (e.g., Paris's 120-day cap), "
        "and personal inconvenience."
    ),
)))

## 3. Host Portfolio — Power Law Analysis

Analyze the distribution of listing counts per host. In marketplace dynamics, host portfolios typically follow a power-law distribution where a small number of 'professional' hosts control a disproportionate share of supply.

In [ ]:
# ── Host portfolio distribution — log-log plot ────────────────────
host_df = db.query("""
    SELECT
        h.host_id,
        h.host_listings_count,
        h.is_professional_host,
        c.display_name AS city
    FROM dim_host h
    JOIN fact_listing_snapshot f ON h.host_key = f.host_key
    JOIN dim_city c ON f.city_key = c.city_key
""")

# Deduplicate hosts (one entry per host per city)
host_unique = host_df.drop_duplicates(subset=['host_id', 'city'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Linear scale
for i, city_name in enumerate(sorted(host_unique['city'].unique())):
    city_hosts = host_unique[host_unique['city'] == city_name]
    counts = city_hosts['host_listings_count'].clip(upper=50)
    axes[0].hist(
        counts, bins=50, alpha=0.6, label=city_name,
        color=AIRBNB_PALETTE[i]
    )

axes[0].set_xlabel('Listings per Host')
axes[0].set_ylabel('Number of Hosts')
axes[0].set_title('Host Portfolio Size Distribution')
axes[0].legend()

# Log-log scale (power law check)
for i, city_name in enumerate(sorted(host_unique['city'].unique())):
    city_hosts = host_unique[host_unique['city'] == city_name]
    counts = city_hosts['host_listings_count'].dropna()
    counts = counts[counts > 0]
    vals, edges = np.histogram(counts, bins=np.logspace(0, np.log10(counts.max()+1), 30))
    centres = (edges[:-1] + edges[1:]) / 2
    mask = vals > 0
    axes[1].scatter(centres[mask], vals[mask], alpha=0.7, s=30,
                    label=city_name, color=AIRBNB_PALETTE[i])

axes[1].set_xscale('log')
axes[1].set_yscale('log')
axes[1].set_xlabel('Listings per Host (log scale)')
axes[1].set_ylabel('Number of Hosts (log scale)')
axes[1].set_title('Power-Law Check: Log-Log Distribution')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Lorenz Curve + Gini Coefficient ───────────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))

gini_results = {}
for i, city_name in enumerate(sorted(host_unique['city'].unique())):
    city_hosts = host_unique[host_unique['city'] == city_name]
    listings = np.sort(city_hosts['host_listings_count'].dropna().values)
    n = len(listings)
    cumulative = np.cumsum(listings) / listings.sum()
    host_pct = np.arange(1, n + 1) / n

    # Gini coefficient
    gini = 1 - 2 * np.trapz(cumulative, host_pct)
    gini_results[city_name] = round(gini, 3)

    ax.plot(host_pct, cumulative, label=f'{city_name} (Gini={gini:.3f})',
            color=AIRBNB_PALETTE[i], linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Perfect equality')
ax.set_xlabel('Cumulative % of Hosts')
ax.set_ylabel('Cumulative % of Listings')
ax.set_title('Lorenz Curve — Host Listing Concentration')
ax.legend(loc='upper left')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

print("\nGini Coefficients (0=perfect equality, 1=one host owns everything):")
for city, gini in gini_results.items():
    print(f"  {city}: {gini}")

In [ ]:
# ── Top host concentration table ──────────────────────────────────
concentration_records = []
for city_name in sorted(host_unique['city'].unique()):
    city_hosts = host_unique[host_unique['city'] == city_name].sort_values(
        'host_listings_count', ascending=False
    )
    total_listings = city_hosts['host_listings_count'].sum()
    total_hosts = len(city_hosts)

    for pct in [1, 5, 10, 20]:
        top_n = max(1, int(total_hosts * pct / 100))
        top_listings = city_hosts.head(top_n)['host_listings_count'].sum()
        concentration_records.append({
            'City': city_name,
            f'Top Hosts %': f'{pct}%',
            'Hosts': top_n,
            'Listings Controlled': int(top_listings),
            'Share of Supply': f'{top_listings / total_listings * 100:.1f}%',
        })

display(pd.DataFrame(concentration_records))

In [ ]:
# ── Business Insight: Power law / concentration ───────────────────
display(Markdown(business_insight(
    title="Hosting Is Dominated by a Small Professional Class",
    finding=(
        "The host portfolio distribution follows a power law: the vast majority of "
        "hosts operate a single listing, while a small professional class (top 5%) "
        "controls a disproportionate share of total supply. "
        f"Gini coefficients are {', '.join(f'{c}: {g}' for c, g in gini_results.items())}."
    ),
    implication=(
        "For regulators: a small number of commercial operators drive a large share "
        "of market activity, potentially reducing housing stock for residents. "
        "For platform operators: professional hosts are critical to supply "
        "reliability but create concentration risk."
    ),
    action=(
        "Investors should monitor the professional-host share as a market maturity "
        "indicator. Regulators considering occupancy caps will disproportionately "
        "impact the top 5-10% of hosts who control the most supply."
    ),
)))

## 4. Review Score Distributions & Rating Inflation

Examine review score distributions and identify the characteristic 'J-curve' pattern that indicates potential rating inflation on the platform.

In [ ]:
# ── Review score distribution — J-curve analysis ──────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Overall rating histogram
rated_df = df[df['review_scores_rating'].notna()]

for i, city_name in enumerate(sorted(rated_df['city'].unique())):
    city_ratings = rated_df[rated_df['city'] == city_name]['review_scores_rating']
    axes[0].hist(
        city_ratings, bins=30, alpha=0.5, label=city_name,
        color=AIRBNB_PALETTE[i], edgecolor='white'
    )

axes[0].set_xlabel('Overall Review Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Review Score Distribution (J-Curve Pattern)')
axes[0].axvline(x=4.5, color='red', linestyle='--', alpha=0.5, label='4.5 threshold')
axes[0].legend()

# Sub-score box plots
sub_scores = [
    'review_scores_accuracy', 'review_scores_cleanliness',
    'review_scores_checkin', 'review_scores_communication',
    'review_scores_location', 'review_scores_value'
]
sub_score_labels = ['Accuracy', 'Cleanliness', 'Check-in',
                    'Communication', 'Location', 'Value']

available_subs = [s for s in sub_scores if s in rated_df.columns and rated_df[s].notna().any()]
sub_data = rated_df[available_subs].melt(var_name='Dimension', value_name='Score')
sub_data['Dimension'] = sub_data['Dimension'].str.replace('review_scores_', '').str.title()

sns.boxplot(
    data=sub_data, x='Dimension', y='Score',
    palette=AIRBNB_PALETTE[:len(available_subs)], ax=axes[1]
)
axes[1].set_title('Review Sub-Score Distributions')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

# Rating inflation statistics
pct_above_4 = (rated_df['review_scores_rating'] >= 4.0).mean() * 100
pct_above_45 = (rated_df['review_scores_rating'] >= 4.5).mean() * 100
print(f"\n{pct_above_4:.1f}% of rated listings score ≥ 4.0")
print(f"{pct_above_45:.1f}% of rated listings score ≥ 4.5")
print(f"Median rating: {rated_df['review_scores_rating'].median():.2f}")

In [ ]:
# ── Rating inflation: Score vs review count ───────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

sample = rated_df.sample(n=min(5000, len(rated_df)), random_state=42)
scatter = ax.scatter(
    sample['number_of_reviews'],
    sample['review_scores_rating'],
    c=[AIRBNB_PALETTE[list(sorted(df['city'].unique())).index(c)] for c in sample['city']],
    alpha=0.3, s=10
)

ax.set_xlabel('Number of Reviews')
ax.set_ylabel('Overall Rating')
ax.set_title('Rating vs Review Volume — Does Experience Inflate Scores?')
ax.set_xscale('log')

# Add city legend manually
for i, city_name in enumerate(sorted(df['city'].unique())):
    ax.scatter([], [], c=AIRBNB_PALETTE[i], label=city_name, s=30)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Business Insight: Rating inflation ────────────────────────────
display(Markdown(business_insight(
    title="Rating Inflation Creates a Compressed Scale",
    finding=(
        f"{pct_above_45:.0f}% of rated listings score 4.5 or above, creating a "
        f"classic 'J-curve' where the effective rating scale runs from ~4.0 to 5.0 "
        f"rather than 1.0 to 5.0. A listing scoring below 4.0 is effectively in "
        f"the bottom tier."
    ),
    implication=(
        "For guests: a 4.2 rating may seem 'good' but is actually below-median "
        "on Airbnb's compressed scale. For platform operators: the current rating "
        "system provides limited discrimination between truly excellent and merely "
        "adequate listings."
    ),
    action=(
        "Hosts should aim for 4.8+ to qualify for Superhost status and maintain "
        "competitive visibility. Guests should interpret any rating below 4.5 "
        "as a meaningful quality signal, not just slightly below average."
    ),
)))

## 5. Availability Patterns

How does average availability vary across cities and seasons?

In [ ]:
# ── Availability patterns by city ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Availability windows comparison
avail_cols = ['availability_30', 'availability_60', 'availability_90', 'availability_365']
avail_labels = ['30 days', '60 days', '90 days', '365 days']

avail_data = []
for city_name in sorted(df['city'].unique()):
    city_df = df[df['city'] == city_name]
    for col, label in zip(avail_cols, avail_labels):
        if col in city_df.columns:
            avail_data.append({
                'City': city_name,
                'Window': label,
                'Avg Available Days': city_df[col].mean(),
            })

avail_df = pd.DataFrame(avail_data)
avail_pivot = avail_df.pivot(index='Window', columns='City', values='Avg Available Days')
avail_pivot = avail_pivot.reindex(['30 days', '60 days', '90 days', '365 days'])
avail_pivot.plot(kind='bar', ax=axes[0], color=AIRBNB_PALETTE[:3])
axes[0].set_title('Average Available Days by Window')
axes[0].set_ylabel('Days')
axes[0].tick_params(axis='x', rotation=0)
axes[0].legend(title='City')

# Occupancy rate distribution
occ_df = df[df['occupancy_rate_pct'].notna()]
sns.boxplot(
    data=occ_df, x='city', y='occupancy_rate_pct',
    palette=AIRBNB_PALETTE[:3], ax=axes[1],
    order=sorted(occ_df['city'].unique())
)
axes[1].set_title('Occupancy Rate Distribution by City')
axes[1].set_ylabel('Occupancy Rate (%)')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

In [ ]:
# ── Monthly availability heatmap from calendar data ───────────────
monthly_avail = db.query("""
    SELECT
        c.display_name AS city,
        d.month_name,
        d.month,
        ROUND(AVG(CASE WHEN fc.is_available THEN 1 ELSE 0 END) * 100, 1)
            AS availability_pct
    FROM fact_calendar fc
    JOIN dim_date d ON fc.date_key = d.date_key
    JOIN fact_listing_snapshot f ON fc.listing_key = f.listing_key
    JOIN dim_city c ON f.city_key = c.city_key
    GROUP BY c.display_name, d.month_name, d.month
    ORDER BY d.month
""")

if len(monthly_avail) > 0:
    pivot = monthly_avail.pivot(index='city', columns='month', values='availability_pct')
    month_names = monthly_avail.drop_duplicates('month').sort_values('month')['month_name'].tolist()
    pivot.columns = month_names[:len(pivot.columns)]

    fig, ax = plt.subplots(figsize=(14, 4))
    sns.heatmap(
        pivot, annot=True, fmt='.1f', cmap='RdYlGn',
        linewidths=0.5, ax=ax, cbar_kws={'label': 'Availability %'}
    )
    ax.set_title('Monthly Availability Rate by City (%)')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Calendar data not available for monthly breakdown")

In [ ]:
# ── Business Insight: Availability patterns ──────────────────────
display(Markdown(business_insight(
    title="Availability Varies Dramatically Across Markets",
    finding=(
        "Average 365-day availability differs significantly between cities, "
        "reflecting different regulatory environments and host strategies. "
        "Monthly patterns reveal seasonal troughs that correlate with peak "
        "tourism periods."
    ),
    implication=(
        "Low availability during peak months signals strong demand and potential "
        "for dynamic pricing. Cities with consistently low availability may face "
        "supply constraints that limit platform growth."
    ),
    action=(
        "Hosts should increase prices during low-availability months (high demand). "
        "Platform operators should focus supply acquisition efforts on "
        "neighbourhoods and months with the tightest supply-demand gaps."
    ),
)))

In [ ]:
# ── Cleanup ───────────────────────────────────────────────────────
db.close()
print("\n✅ Notebook 01 complete — Summary Statistics & Distributions")